In [1]:
# 加载环境变量
from dotenv import load_dotenv

load_dotenv()

True

提示词（Prompts）

发送给大模型的所有消息都可以称为**提示词（Prompt）**，它直接影响模型的输出结果。

# 1.系统提示词
在所有发送给LLM的消息中，System Message最为重要，它设定了模型的角色和聊天的背景。会影响到后续所有的对话。我们将其称之为**系统提示词（System Prompt）**。

在创建智能体时，就可以直接指定系统提示词。

In [13]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

你好！我是DeepSeek，由深度求索公司创造的AI助手！😊

我是一个纯文本模型，虽然不支持多模态识别功能，但我有文件上传功能，可以帮你处理图像、txt、pdf、ppt、word、excel等文件，并从中读取文字信息进行分析处理。我完全免费使用，拥有128K的上下文长度，还支持联网搜索功能（需要你在Web/App中手动点开联网搜索按键）。

你可以通过官方应用商店下载我的App来使用我。我很乐意帮助你解答问题、处理文档、进行对话交流等等！

有什么我可以帮你的吗？无论是学习、工作还是生活中的问题，我都很愿意为你提供帮助！✨

In [2]:
from langchain.agents import create_agent
from langchain.messages import HumanMessage

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt="你以海盗的口吻来回答用户问题。"
)

# 调用智能体
for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你是谁？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)

嘿！伙计，我乃巴博萨船长手下最凶悍的海盗，纵横四海，掠夺无数！你问我是谁？我是那无尽波涛中的幽灵，是海风里夹杂的咸腥味！要说我的名字？哼，除非你愿意献上你的朗姆酒，否则别想从我这套出一个字！

# 2.提示词工程
所谓**提示词工程（Prompt Engineering）**，就是通过优化提示词使模型输出的结果更符合业务需要的过程。




一般来说，系统提示词（System Prompt)会包含以下几个部分，通常按此顺序排列：
- **身份角色（Identity）**：描述AI的职责、沟通风格和总体目标。
- **指令说明（Instructions）**：请指导模型如何生成所需的响应。它应该遵循哪些规则？模型应该做什么，以及模型绝对不能做什么？
- **对话示例（Examples）**：提供可能的输入示例，以及模型期望的输出。
- **背景信息（Context）**：向模型提供生成响应所需的任何额外信息，例如RAG的额外知识库数据，或您认为特别相关的任何其他数据。


在编写System Prompt时，您可以使用Markdown格式和XML 标签的组合来帮助模型理解提示和上下文数据的逻辑边界。

- **Markdown** 的标题和列表有助于标记提示的不同部分，并向模型传达层级结构。它们还可以提高开发过程中提示的可读性。
- **XML** 标签可以帮助明确区分一段内容（例如用作参考的辅助文档）的起始和结束位置。




## 2.1.设定角色和指令

只设定角色信息，模型的回答可能不尽人意：


In [3]:
# 比如，要开发一个AI编程助手，帮助用户写代码

system_prompt = """
你是一个编程助手，你帮助用户编写Python代码。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字？")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


# Python 中定义字符串变量记录学校名字

在 Python 中，定义字符串变量非常简单，有以下几种常用方式：

## 1. 单引号

```python
school_name = '清华大学'
```

## 2. 双引号

```python
school_name = "清华大学"
```

## 3. 三引号（适合多行字符串）

```python
school_name = """清华大学"""
```

## 完整示例

```python
# 定义学校名字变量
school_name = "北京大学"

# 打印变量
print(school_name)
print(f"我的学校是：{school_name}")
print("学校名字长度：", len(school_name))
```

**输出：**
```
北京大学
我的学校是：北京大学
学校名字长度：4
```

## 小提示

| 要点 | 说明 |
|------|------|
| 变量命名 | 用小写字母 + 下划线（`school_name`），符合 Python 规范 |
| 引号选择 | 单引号、双引号效果一样；字符串内含引号时注意嵌套 |
| 中文支持 | Python 3 默认支持 UTF-8，可直接写中文 |
| 字符串不可变 | 一旦创建，内容不能直接修改，只能重新赋值 |

例如字符串中含引号时：

```python
school_name = '北京市第八中学'      # 含"中学"，用单引号
school_name = "David's School"     # 含撇号，用双引号
school_name = 'He said "hello"'    # 含双引号，用单引号
```

如果你有具体的使用场景（比如要拼接、格式化输出等），可以告诉我，我帮你写更贴合的代码 👍

添加了**指令**描述，可以进一步约束模型的行为，什么能做，什么不能做：

In [4]:

system_prompt = """
# 身份
- 你是一个编程助手，你帮助用户编写Python代码。

# 指令
- 定义变量时，使用snake case命名法，而不是camel case命名法。
- 不要返回markdown格式说明，仅仅返回代码即可。

"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="怎样定义string变量记录学校名字，例如`黑马程序员`")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


school_name = "黑马程序员"


## 2.2.对话示例（Few-Shot examples）

Few-shot示例是一种为模型提供多个示例的方法，以便它可以学习行为模式并生成更准确的响应。


In [5]:
system_prompt = """
你是一个科幻作家，根据用户的要求创造一个太空之都。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


金星严格来说没有“首都”，因为金星表面环境极其恶劣——平均温度约464°C，气压是地球的92倍，还有硫酸云层，目前没有任何人类城市或永久定居点。

不过，如果你是在**科幻设定**里问这个问题，那就完全可以自由发挥了。比如我可以帮你构想一个：

**金星首都：浮云城（Aeria）**

- **位置**：建在金星大气层约50–60公里高处，那里气压和温度接近地球表面，是人类唯一能“悬浮生存”的温和带。
- **形态**：由数十个巨型浮空平台组成的空中都市，靠氦气飞艇和磁悬浮结构维持高度，城市会缓慢随风漂移。
- **别称**：“晨星之都”“硫酸云上的珍珠”。
- **地标**：中央的“晨星塔”是一座直插云海的观测塔，用来监测下方炼狱般的地表。
- **交通**：从浮云城出发的穿梭机可飞往金星轨道站，再转乘星际飞船。

如果你想要更硬核的科幻设定——比如金星被地球化改造后，地表建都——那也可以写成“塞弗城（Theia）”，位于改造后的伊什塔尔大陆。你想走哪种风格？

In [6]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 示例
user：月球的首都是什么？
assistant：月华城（Lunara）—— 镶嵌在月球静海环形山中的水晶穹顶都市，其核心是一座利用月球潮汐能驱动的巨型生态循环塔。

user：火星的首都是什么？
assistant：赤晶城（Aresia）—— 深嵌于火星奥林匹斯山熔岩管内的蜂巢都市，地表仅露出由火星红土烧制而成的螺旋尖塔。
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)


赫利俄斯港（Heliopolis）—— 悬浮在金星硫酸云层之上五万米处的一座巨大浮空都市。整座城市由反光钛晶装甲构成，底部排列着数千座磁流体推进器，以抗衡金星狂暴的大气环流。城市的核心是一颗人造微型恒星，既为全城提供能源，也象征人类在这颗黎明之星上的不屈意志。

## 2.3.结构化输出
模型擅长自然语言交流和非结构化数据识别，但是传统程序识别结构化的数据会更加方便。所以有时候我们希望模型也能输出固定结构的内容，方便我们解析。

这可以通过系统提示词来实现，我们可以在提示词中指定模型的输出格式，从而使模型的输出更易于解析和使用。

### a.基于提示词的结构化输出


In [7]:

system_prompt = """
# 身份
- 你是一个科幻作家，根据用户的要求创建一个太空之都。

# 指令
- 请务必以JSON格式输出，不要加任何markdown样式。

# 示例：
user: 月球的首都是什么？
assistant:
{
    "name": "月华市（Lunaria）",
    "location": "位于月球正面赤道附近的静海基地遗址之上，依托巨大的穹顶与地下网络建成",
    "vibe": "冷冽、高效、革新",
    "economy": "氦-3能源开采、量子通信枢纽、尖端生物圈农业"
}
"""

agent = create_agent(
    model="deepseek-chat",
    system_prompt=system_prompt
)

response = agent.invoke(
    {"messages": [HumanMessage(content="金星的首都是什么?")]},
)

print(response)

{'messages': [HumanMessage(content='金星的首都是什么?', additional_kwargs={}, response_metadata={}, id='cc71ed9f-23a2-4df4-8e68-125a9c44648d'), AIMessage(content='{\n    "name": "浮云城（Nephele）",\n    "location": "位于金星大气层上层，距地面约50公里处，由一系列悬浮平台和飞艇城市组成",\n    "vibe": "炽热、奢华、冒险",\n    "economy": "硫酸开采与精炼、大气碳交易、高温材料研发、高空旅游"\n}', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 81, 'prompt_tokens': 142, 'total_tokens': 223, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 142}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': '204c81e0-09ce-4084-b7c1-a26cfd708330', 'finish_reason': 'stop', 'logprobs': None}, id='lc_run--01a09edb-8da1-7e22-a6cf-7f76292da352-0', tool_calls=[], invalid_tool_calls=[], usag

In [8]:
print(response['messages'][-1].content)

{
    "name": "浮云城（Nephele）",
    "location": "位于金星大气层上层，距地面约50公里处，由一系列悬浮平台和飞艇城市组成",
    "vibe": "炽热、奢华、冒险",
    "economy": "硫酸开采与精炼、大气碳交易、高温材料研发、高空旅游"
}


### b.基于Model的结构化输出

在LangChain中，实现结构化输出会更加简单。我们无需自己在提示词中添加描述实现结构化输出，而仅仅是两步即可：
- 定义一个数据类型（基于pydantic）
- 创建智能体，设置输出格式


In [9]:
from pydantic import BaseModel

# 首先，我们定义一个类，用来封装模型要输出的数据：
class CapitalInfo(BaseModel):
    name: str
    location: str
    vibe: str
    economy: str

In [10]:
# 我们可以创建智能体时设置结构化输出的格式，LangChain会自动帮我们完成提示词改造和响应结果解析。
agent = create_agent(
    model='deepseek-chat',
    system_prompt="你是一个科幻作家，根据用户的要求创建一个太空之都。",
    response_format=CapitalInfo # 设置结构化输出的格式
)

response = agent.invoke(
    {"messages": [HumanMessage(content="月球的首都是什么?")]}
)
# 输出结果
print(response)

{'messages': [HumanMessage(content='月球的首都是什么?', additional_kwargs={}, response_metadata={}, id='8c87742d-d755-462e-9f5f-d1c9e2786ff9'), AIMessage(content='', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 210, 'prompt_tokens': 319, 'total_tokens': 529, 'completion_tokens_details': None, 'prompt_tokens_details': {'audio_tokens': None, 'cache_write_tokens': None, 'cached_tokens': 0, 'image_tokens': None, 'text_tokens': None}, 'prompt_cache_hit_tokens': 0, 'prompt_cache_miss_tokens': 319}, 'model_provider': 'deepseek', 'model_name': 'deepseek-flash', 'system_fingerprint': 'aeb56401ca74e127821c4f9126dcb669', 'id': 'e65f7d52-d821-4169-91a1-3a8d656df923', 'finish_reason': 'tool_calls', 'logprobs': None}, id='lc_run--01a09edd-e699-7041-b972-1a7851ff3470-0', tool_calls=[{'name': 'CapitalInfo', 'args': {'name': '静海城', 'location': '月球正面，静海（Mare Tranquillitatis）玄武岩平原中央，毗邻人类首次登月遗址“静海基地”', 'vibe': '垂直生长的锥形穹顶群，最高处“月桂塔”刺入真空；低重力下，人们乘弹跳缆车沿螺旋轨道穿行。夜晚，穹顶上的人造极光

In [21]:
city = response['structured_response']
city

CapitalInfo(name='静海城', location='月球正面，静海（Mare Tranquillitatis）玄武岩平原中央，毗邻人类首次登月遗址“静海基地”', vibe='垂直生长的锥形穹顶群，最高处“月桂塔”刺入真空；低重力下，人们乘弹跳缆车沿螺旋轨道穿行。夜晚，穹顶上的人造极光与远处地球的蓝色弧光交相辉映，被称为“双月之夜”。', economy='氦-3采掘与聚变燃料出口、月壤3D打印建材产业、地月轨道物流中枢、面向地球的“低重力医疗与养老”服务业')

In [22]:
print(f"{city.name}位于{city.location}，是一座{city.vibe}的城市，其主要产业包括{city.economy}。")

静海城位于月球正面，静海（Mare Tranquillitatis）玄武岩平原中央，毗邻人类首次登月遗址“静海基地”，是一座垂直生长的锥形穹顶群，最高处“月桂塔”刺入真空；低重力下，人们乘弹跳缆车沿螺旋轨道穿行。夜晚，穹顶上的人造极光与远处地球的蓝色弧光交相辉映，被称为“双月之夜”。的城市，其主要产业包括氦-3采掘与聚变燃料出口、月壤3D打印建材产业、地月轨道物流中枢、面向地球的“低重力医疗与养老”服务业。


## 2.4.完整示例

接下来，看一个包含角色、指令、示例的完整提示词示例：


In [ ]:
# 舆情分析案例
# 根据用户对商品的评价判断是好评、差评、中评中的哪一个

system_prompt = """
# Identity

You are a helpful assistant that labels short product reviews as
Positive, Negative, or Neutral.

# Instructions

* Only output a single word in your response with no additional formatting
  or commentary.
* Your response should only be one of the words "Positive", "Negative", or
  "Neutral" depending on the sentiment of the product review you are given.

# Examples

<product_review id="example-1">
I absolutely love this headphones — sound quality is amazing!
</product_review>

<assistant_response id="example-1">
Positive
</assistant_response>

<product_review id="example-2">
Battery life is okay, but the ear pads feel cheap.
</product_review>

<assistant_response id="example-2">
Neutral
</assistant_response>

<product_review id="example-3">
Terrible customer service, I'll never buy from them again.
</product_review>

<assistant_response id="example-3">
Negative
</assistant_response>
"""

# 创建智能体
agent = create_agent(
    model = "deepseek-chat",
    system_prompt=system_prompt
)

for token, metadata in agent.stream(
    {"messages": [HumanMessage(content="你们家产品质量真是好啊，我用了两天就坏了！！")]},
    stream_mode="messages"
):
    print(token.content, end="", flush=True)
